# DeepSense 6G V2V Scenario 36: Fully Annotated Dataset Walkthrough
### Complete Multi-Modal Sensor Specification, Feature Engineering, and Codebook Analysis
---
This interactive notebook provides an exhaustive inspection of the real **DeepSense 6G Scenario 36** dataset (24,799 samples across 124 continuous vehicle drives).

In [ ]:
# Cell 1: Imports and Global Setup
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Loading scenario36.p and scenario36.csv...")
with open("scenario36.p", "rb") as f:
    pdata = pickle.load(f)

csv_df = pd.read_csv("scenario36.csv")
print(f"Pickle Keys: {len(pdata.keys())} keys | CSV Shape: {csv_df.shape}")

## 1. Dataset Schema & Hardware Mapping
- **Unit 1 (Receiver)**: Ford Transit van equipped with four 64-element 60 GHz mmWave phased arrays, forward-facing ZED2 RGB stereo camera, and RTK-GPS.
- **Unit 2 (Transmitter)**: Lincoln MKZ vehicle transmitting an omnidirectional 60 GHz beacon signal with synchronized RTK-GPS.

In [ ]:
# Cell 2: Feature Inspection & DataFrame Assembly
df = pd.DataFrame({
    "abs_index": pdata["abs_index"],
    "seq_index": pdata["seq_index"],
    "unit1_gps1_lat": pdata["unit1_gps1_lat"],
    "unit1_gps1_lon": pdata["unit1_gps1_lon"],
    "unit2_gps1_lat": pdata["unit2_gps1_lat"],
    "unit2_gps1_lon": pdata["unit2_gps1_lon"],
    "unit1_overall-beam": pdata["unit1_overall-beam"]
})

# Extract 256 Power Vectors
pwr1 = np.vstack(pdata["unit1_pwr1"])
pwr2 = np.vstack(pdata["unit1_pwr2"])
pwr3 = np.vstack(pdata["unit1_pwr3"])
pwr4 = np.vstack(pdata["unit1_pwr4"])
pwr_all = np.hstack([pwr1, pwr2, pwr3, pwr4])

print(f"Total Samples: {len(df):,} | Power Matrix Shape: {pwr_all.shape}")
df.head(5)

## 2. 12 Spatial Kinematic Features Formulation
The following 12 spatial kinematic features are engineered from raw GPS coordinates to capture vehicle-to-vehicle trajectory dynamics:

In [ ]:
# Cell 3: 12 Kinematic Features Engineering
rel_lat = df["unit2_gps1_lat"] - df["unit1_gps1_lat"]
rel_lon = df["unit2_gps1_lon"] - df["unit1_gps1_lon"]
distance = np.sqrt(rel_lat**2 + rel_lon**2) * 111000.0
bearing = np.arctan2(rel_lon, rel_lat).astype(np.float32)

kin_df = pd.DataFrame({
    "rel_lat": rel_lat,
    "rel_lon": rel_lon,
    "distance_m": distance,
    "bearing_rad": bearing,
    "sin_bearing": np.sin(bearing),
    "cos_bearing": np.cos(bearing),
    "bearing_rate": np.gradient(bearing),
    "distance_rate": np.gradient(distance),
    "unit1_speed": pdata.get("unit1_gps1_speed", np.zeros(len(df))),
    "unit1_heading": pdata.get("unit1_gps1_heading", np.zeros(len(df))),
    "unit2_speed": pdata.get("unit2_gps1_speed", np.zeros(len(df))),
    "unit2_heading": pdata.get("unit2_gps1_heading", np.zeros(len(df)))
})

kin_df.describe().T[["mean", "std", "min", "50%", "max"]]

## 3. Visualizing 256-Element Beam Spectra & Codebook Geometry

In [ ]:
# Cell 4: 256-Beam Power Profile Visualization
sample_idx = 100
pwr_sample_linear = pwr_all[sample_idx]
pwr_sample_db = 10.0 * np.log10(pwr_sample_linear + 1e-12)
true_best_beam = df["unit1_overall-beam"].iloc[sample_idx]

plt.figure(figsize=(10, 3.5), dpi=150)
plt.bar(range(256), pwr_sample_db, color='#2563EB', width=0.8, alpha=0.8, label='Beam Power (dBm)')
plt.axvline(true_best_beam, color='red', linestyle='--', linewidth=2, label=f'Optimal Beam (Index {true_best_beam})')
plt.title(f'Sample #{sample_idx}: 256-Element Beam Power Profile Across 4 Subarrays', fontsize=11, fontweight='bold')
plt.xlabel('Beam Codebook Index (0 - 255)', fontsize=9)
plt.ylabel('Received Power (dBm)', fontsize=9)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 4. Drive-Block Splitting (Zero-Leakage Architecture)
Splitting samples strictly by `seq_index` (124 continuous vehicle drives) guarantees zero spatial or temporal data leakage between Train (70%), Calibration (15%), and Test (15%) sets.

In [ ]:
# Cell 5: Drive-Block Split Statistics
drives = df["seq_index"].unique()
rng = np.random.default_rng(42)
rng.shuffle(drives)

n_tr = int(len(drives) * 0.70)
n_ca = int(len(drives) * 0.15)

train_drives = set(drives[:n_tr])
calib_drives = set(drives[n_tr:n_tr + n_ca])
test_drives = set(drives[n_tr + n_ca:])

n_train = len(df[df["seq_index"].isin(train_drives)])
n_calib = len(df[df["seq_index"].isin(calib_drives)])
n_test = len(df[df["seq_index"].isin(test_drives)])

print(f"Train Set:       {len(train_drives)} drives | {n_train:,} samples (70.0%)")
print(f"Calibration Set: {len(calib_drives)} drives | {n_calib:,} samples (15.0%)")
print(f"Test Set:        {len(test_drives)} drives | {n_test:,} samples (15.0%)")